# CLM-8B System One Model

This notebook shows how to use the [generic-server](../../../../charts/machine-learning/serving/generic-server/) Helm chart to serve [CLM-8B](https://github.com/Contrastive-LM/CLM), a System One model that answers typed questions about a state with probabilities.

It runs two releases: `clm-encoder` (Qwen3-8B as a vLLM pooling model on a g5.2xlarge) and `clm-serve` (the CLM heads and API on CPU). See [README.md](./README.md) for the design and the caveats.

## Setup and Imports

In [ ]:
! pip install kubernetes
! pip install boto3
! pip install -r ~/amazon-eks-machine-learning-with-terraform-and-kubeflow/examples/inference/system-one/clm-8b/example/requirements.txt

In [ ]:
import json
import os
import subprocess
import sys

# Set working directory
os.chdir(os.path.expanduser('~/amazon-eks-machine-learning-with-terraform-and-kubeflow'))
print(f"Working directory: {os.getcwd()}")

# Get the src directory
src_dir = os.path.join(os.getcwd(), "src")
sys.path.insert(0, src_dir)

from k8s.utils import (
    wait_for_helm_release_pods,
    wait_for_triton_server,
    find_k8s_service
)

# Get notebook directory
notebook_dir = os.path.join(os.getcwd(), 'examples', 'inference', 'system-one', 'clm-8b')
print(f"Notebook directory: {notebook_dir}")

# initialize key variables. clm-serve calls the encoder by its release name, so keep 'clm-encoder'.
encoder_release = 'clm-encoder'
serve_release = 'clm-serve'
namespace = 'kubeflow-user-example-com'

## Step 1: Build and Push Docker Containers

The encoder uses the repository's `ray-pytorch-vllm` container; `clm-serve` uses the small CPU container in [containers/clm-serve](../../../../containers/clm-serve/). The region is detected from your AWS configuration.

In [ ]:
import boto3

# Create a Boto3 session
session = boto3.session.Session()

# Access the region_name attribute to get the current region
current_region = session.region_name

def run_streaming(cmd):
    # Start the subprocess with streaming output
    process = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                               text=True, bufsize=1, universal_newlines=True)
    for line in process.stdout:
        print(line, end='')
        sys.stdout.flush()
    return_code = process.wait()
    if return_code != 0:
        print(f"\nProcess exited with return code: {return_code}")
    else:
        print("\nProcess completed successfully")
    return return_code

for container in ['ray-pytorch-vllm', 'clm-serve']:
    assert run_streaming([f'./containers/{container}/build_tools/build_and_push.sh', current_region]) == 0

In [ ]:
# The image URIs are passed to helm with --set, so they never live in the values files.
def image_uri(container: str) -> str:
    env = subprocess.run(['bash', '-c', f'source ./containers/{container}/build_tools/set_env.sh && echo $IMAGE_NAME:$IMAGE_TAG'],
                         capture_output=True, text=True, check=True).stdout.strip()
    account = boto3.client('sts').get_caller_identity()['Account']
    return f"{account}.dkr.ecr.{current_region}.amazonaws.com/{env}"

encoder_image = image_uri('ray-pytorch-vllm')
serve_image = image_uri('clm-serve')

## Step 2: Download the Model Weights

Qwen3-8B and the CLM-8B heads are public, so the Hugging Face token is optional.

In [ ]:
# Optional: set your Hugging Face token
hf_token = None

downloads = {'clm-qwen3-8b': 'Qwen/Qwen3-8B', 'clm-heads': 'Contrastive-LM/CLM-v0.1-8B'}

for release, hf_model_id in downloads.items():
    env = [{"name": "HF_MODEL_ID", "value": hf_model_id}]
    if hf_token:
        env.append({"name": "HF_TOKEN", "value": hf_token})
    cmd = [
        'helm', 'install', '--debug', release,
        'charts/machine-learning/model-prep/hf-snapshot',
        '--set-json', f'env={json.dumps(env)}',
        '-n', namespace
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

In [ ]:
# wait for model download jobs to complete
for release in downloads:
    assert wait_for_helm_release_pods(release_name=release, namespace=namespace)

In [ ]:
# Uninstall the Helm charts after completion
for release in downloads:
    cmd = ['helm', 'uninstall', release, '-n', namespace]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

## Step 3: Launch the Encoder and clm-serve

The encoder loads Qwen3-8B from FSx, which takes a few minutes after its node is up.

In [ ]:
for release, values, image in [(encoder_release, 'clm-encoder.yaml', encoder_image),
                               (serve_release, 'clm-serve.yaml', serve_image)]:
    cmd = [
        'helm', 'install', '--debug', release,
        'charts/machine-learning/serving/generic-server',
        '-f', f'{notebook_dir}/{values}',
        '--set', f'image.name={image}',
        '-n', namespace
    ]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)

In [ ]:
# Wait for both pods to be Running and Ready
for release in [encoder_release, serve_release]:
    assert wait_for_triton_server(release_name=release, namespace=namespace)

## Step 4: Check Service Status

In [ ]:
service = find_k8s_service(serve_release, namespace)
if service:
    print(f"Service {service.metadata.name} is available.")
    print(f"Service type: {service.spec.type}")
    print(f"Service ports: {service.spec.ports} ")
    print(f"Run\n\n kubectl port-forward -n {namespace} svc/{service.metadata.name}  8700:8700 \n\nin a new terminal")
else:
    print(f"Service {serve_release} not found.")

In [ ]:
import time
import requests

service_url = "http://localhost:8700"  # Adjust based on your setup

# /health returns 200 even while the encoder is unreachable, so wait for "embedder": true.
for _ in range(60):
    try:
        health = requests.get(f"{service_url}/health", timeout=5).json()
        if health.get("embedder"):
            break
        print("clm-serve is up, waiting for the encoder...")
    except requests.RequestException as e:
        print(f"Waiting for clm-serve: {e}")
    time.sleep(10)
print(health)
assert health.get("embedder"), "the encoder is not reachable from clm-serve"

## Step 5: Parity Check

The reference answers below come from [parity_reference.py](./parity_reference.py), which computes the Qwen3-8B embeddings with Hugging Face Transformers instead of vLLM and runs them through the same `contrastive-lm` 0.1.0 engine and CLM-v0.1-8B heads. Matching them confirms that the encoder, its pooling and its truncation are what the heads expect. A `CHECK` here means the numbers that follow are not CLM-8B's.

In [ ]:
r = requests.post(f"{service_url}/v1/systemone", json={
    "state": "Customer: my invoice was charged twice and nobody answers the phone!",
    "model": "clm-latest",
    "questions": {
        "urgency": {"type": "noul", "instructions": "Is this urgent?"},
        "department": {"type": "choice", "instructions": "Which team should handle this?",
                       "criteria": {"billing": "Charges, invoices, refunds",
                                    "technical": "Bugs and outages"}},
        "frustration": {"type": "score", "instructions": "How frustrated is the customer?",
                        "criteria": ["Calm", "Frustrated", "Very angry"]},
    },
})
r.raise_for_status()
answers = r.json()["answers"]

observed = {
    "urgency": answers["urgency"]["noul"],
    "billing": answers["department"]["probabilities"]["billing"],
    "frustration": answers["frustration"]["score"],
}
# From parity_reference.py: contrastive-lm 0.1.0, CLM-v0.1-8B, Qwen3-8B bf16 via Transformers.
reference = {"urgency": 0.84939, "billing": 0.98830, "frustration": 1.99998}
for k, ref in reference.items():
    status = "PASS" if abs(observed[k] - ref) <= 0.02 else "CHECK"
    print(f"{status}  {k:12s} observed={observed[k]:.5f}  reference={ref:.5f}")

## Step 6: Rank Candidates

`/v1/rank` scores free-form answers against a context and a question, best first. It is useful for tool selection or for picking among best-of-N generations.

In [ ]:
r = requests.post(f"{service_url}/v1/rank", json={
    "context": "User: book me a table for two tomorrow at 7pm",
    "question": "Which tool should the agent call first?",
    "answers": ["search_restaurants", "create_calendar_event", "send_email"],
})
r.raise_for_status()
for item in r.json()["ranked"]:
    print(f"{item['rank']}. {item['candidate']:24s} {item['prob']:.3f}")

## Step 7: Load Test

Sends 96 new states at each concurrency level, three questions per request, with no state-cache hits, and checks that every request completes. After each level, it also checks that the encoder holds no requests (`vllm:num_requests_running` is 0). It records request and error counts in `example/results/loadtest.json`, not latency or throughput. See [example/README.md](./example/README.md).

The load test runs as a Kubernetes Job next to the services, not through `kubectl port-forward`: the port-forward tunnel fails under concurrent connections, and its errors would hide the server's.

In [ ]:
import time

from kubernetes import client, config

config.load_kube_config()
core, batch = client.CoreV1Api(), client.BatchV1Api()

# The load test runs inside the cluster, next to the services. Through kubectl port-forward,
# concurrent connections break the tunnel and the errors would measure the tunnel, not clm-serve.
job_name = 'clm-loadtest'
example_files = ['loadtest.py', 'states.jsonl']
core.create_namespaced_config_map(namespace, client.V1ConfigMap(
    metadata=client.V1ObjectMeta(name=job_name),
    data={f: open(f'{notebook_dir}/example/{f}').read() for f in example_files}))

script = ("cp -r /example /tmp/example && cd /tmp/example && "
          "python loadtest.py --out /tmp/example/results --url http://clm-serve:8700 --encoder-metrics-url http://clm-encoder:8000/metrics "
          "--concurrency 1,2,4,8,16,32 && echo '=== loadtest.json' && cat /tmp/example/results/loadtest.json")
batch.create_namespaced_job(namespace, {
    'apiVersion': 'batch/v1', 'kind': 'Job', 'metadata': {'name': job_name},
    'spec': {'backoffLimit': 0, 'template': {
        'metadata': {'annotations': {'sidecar.istio.io/inject': 'false', 'karpenter.sh/do-not-disrupt': 'true'}},
        'spec': {'restartPolicy': 'Never',
                 'tolerations': [{'key': 'compute-tier', 'operator': 'Exists', 'effect': 'NoSchedule'}],
                 'volumes': [{'name': 'example', 'configMap': {'name': job_name}}],
                 'containers': [{'name': 'loadtest', 'image': serve_image, 'command': ['bash', '-c', script],
                                 'volumeMounts': [{'name': 'example', 'mountPath': '/example'}],
                                 'resources': {'requests': {'cpu': '1', 'memory': '1Gi'},
                                               'limits': {'cpu': '2', 'memory': '2Gi'}}}]}}}})

while True:
    status = batch.read_namespaced_job_status(job_name, namespace).status
    if status.succeeded or status.failed:
        break
    time.sleep(10)
pod = core.list_namespaced_pod(namespace, label_selector=f'job-name={job_name}').items[0].metadata.name
log = core.read_namespaced_pod_log(pod, namespace, _preload_content=False).data.decode()
print(log.split('=== loadtest.json')[0])

batch.delete_namespaced_job(job_name, namespace, propagation_policy='Background')
core.delete_namespaced_config_map(job_name, namespace)
assert status.succeeded, "the load-test job failed; see the log above"

os.makedirs(f'{notebook_dir}/example/results', exist_ok=True)
with open(f'{notebook_dir}/example/results/loadtest.json', 'w') as f:
    f.write(log.split('=== loadtest.json')[1])
load = json.load(open(f'{notebook_dir}/example/results/loadtest.json'))
assert all(level['errors'] == 0 for level in load['levels']), "some requests failed or timed out"
assert all(level['encoder_running_after'] == 0 for level in load['levels']), "the encoder still holds requests"

## Step 8: Stop the Services

In [ ]:
for release in [serve_release, encoder_release]:
    cmd = ['helm', 'uninstall', release, '-n', namespace]
    result = subprocess.run(cmd, capture_output=True, text=True)
    print(result.stdout)
    if result.stderr:
        print("STDERR:", result.stderr)